# ESCO Career Recommendation Dataset Preprocessing - Milestone 2

Upload either the ESCO raw ZIP file or the three CSV files: occupations, skills, occupationSkillRelations.
Then run the preprocessing pipeline.


In [ ]:
# Run this in Google Colab first if your files are not already uploaded
from google.colab import files
uploaded = files.upload()


In [ ]:
# ============================================================
# ESCO Career Recommendation Dataset Preprocessing - Milestone 2
# Works with either:
#   1) ESCO raw ZIP file containing occupations_en.csv, skills_en.csv, occupationSkillRelations_en.csv
#   2) Raw CSV files uploaded separately
#   3) Your partially preprocessed files with columns like occupation_title, skill_title, etc.
# Output: cleaned files + combined career_roles_prepared.csv ready for embeddings.
# ============================================================

import os
import re
import json
import glob
import zipfile
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -------------------------
# CONFIG
# -------------------------
OUTPUT_DIR = "processed_esco"
EDA_DIR = os.path.join(OUTPUT_DIR, "eda_outputs")
MAX_ESSENTIAL_SKILLS_FOR_CONTEXT = 80
MAX_OPTIONAL_SKILLS_FOR_CONTEXT = 50
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(EDA_DIR, exist_ok=True)

# -------------------------
# Helpers
# -------------------------
def clean_text(x):
    if pd.isna(x):
        return ""
    x = str(x)
    x = x.replace("\r", " ").replace("\n", "; ")
    x = re.sub(r"\s+", " ", x).strip()
    return x


def clean_label(x):
    x = clean_text(x)
    return x.strip()


def normalize_key(x):
    x = clean_label(x).lower()
    x = re.sub(r"[^a-z0-9+#. ]+", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x


def uri_short_id(uri):
    uri = clean_text(uri)
    if not uri:
        return ""
    return uri.rstrip("/").split("/")[-1]


def make_slug_id(prefix, label):
    label = normalize_key(label)
    label = re.sub(r"[^a-z0-9]+", "_", label).strip("_")
    return f"{prefix}_{label[:80]}" if label else f"{prefix}_missing"


def first_existing(columns, choices):
    for c in choices:
        if c in columns:
            return c
    return None


def find_input_file(keyword):
    files = glob.glob(f"*{keyword}*.csv")
    if not files:
        return None
    # Prefer non-output and raw-ish files if multiple exist
    files = [f for f in files if not f.startswith(OUTPUT_DIR)]
    return sorted(files)[0] if files else None


def load_esco_files():
    """Load ESCO files from ZIP if present, else from CSVs in current directory."""
    zip_candidates = glob.glob("*.zip")
    esco_zip = None
    for z in zip_candidates:
        try:
            with zipfile.ZipFile(z) as zf:
                names = set(zf.namelist())
                if {"occupations_en.csv", "skills_en.csv", "occupationSkillRelations_en.csv"}.issubset(names):
                    esco_zip = z
                    break
        except Exception:
            pass

    if esco_zip:
        print(f"Loading raw ESCO files from ZIP: {esco_zip}")
        with zipfile.ZipFile(esco_zip) as zf:
            occ = pd.read_csv(zf.open("occupations_en.csv"))
            skills = pd.read_csv(zf.open("skills_en.csv"))
            rel = pd.read_csv(zf.open("occupationSkillRelations_en.csv"))
        source_type = "raw_zip"
    else:
        occ_path = find_input_file("occupations")
        skill_path = find_input_file("skills")
        rel_path = find_input_file("occupationSkillRelations")
        if not occ_path or not skill_path or not rel_path:
            raise FileNotFoundError(
                "Upload either the ESCO ZIP file or the three CSV files: occupations, skills, occupationSkillRelations."
            )
        print("Loading CSV files:")
        print("-", occ_path)
        print("-", skill_path)
        print("-", rel_path)
        occ = pd.read_csv(occ_path)
        skills = pd.read_csv(skill_path)
        rel = pd.read_csv(rel_path)
        source_type = "csv_files"
    return occ, skills, rel, source_type


# -------------------------
# Review raw/current files
# -------------------------
def review_df(df, name):
    print(f"\n===== {name} =====")
    print("Shape:", df.shape)
    print("Columns:", list(df.columns))
    print("Missing values:")
    print(df.isna().sum())
    print("Duplicate full rows:", df.duplicated().sum())
    display(df.head(3)) if 'display' in globals() else print(df.head(3))


# -------------------------
# Standardize occupations
# -------------------------
def preprocess_occupations(occ_raw):
    occ = occ_raw.copy()
    cols = occ.columns

    concept_col = first_existing(cols, ["conceptUri", "occupationUri", "uri"])
    title_col = first_existing(cols, ["preferredLabel", "occupation_title", "occupationLabel", "title"])
    alt_col = first_existing(cols, ["altLabels", "alt_occupation_title", "alternativeLabels"])
    desc_col = first_existing(cols, ["description", "occ_description"])
    isco_col = first_existing(cols, ["iscoGroup", "isco_group"])
    code_col = first_existing(cols, ["code", "occupation_code"])

    if title_col is None:
        raise ValueError("Occupation title column not found.")

    out = pd.DataFrame()
    out["conceptUri"] = occ[concept_col].apply(clean_text) if concept_col else ""
    out["occupation_title"] = occ[title_col].apply(clean_label)
    out["alt_occupation_title"] = occ[alt_col].apply(clean_text) if alt_col else ""
    out["occ_description"] = occ[desc_col].apply(clean_text) if desc_col else ""
    out["isco_group"] = occ[isco_col].apply(clean_text) if isco_col else ""
    out["occupation_code"] = occ[code_col].apply(clean_text) if code_col else ""

    # If raw file lacks ESCO URI, create a stable local URI using title.
    missing_uri = out["conceptUri"].eq("")
    out.loc[missing_uri, "conceptUri"] = out.loc[missing_uri, "occupation_title"].apply(
        lambda x: "local://occupation/" + make_slug_id("occ", x)
    )

    out["occupation_id"] = out["conceptUri"].apply(uri_short_id)
    out["occupation_key"] = out["occupation_title"].apply(normalize_key)

    # Drop exact duplicate conceptUri rows
    out = out.drop_duplicates(subset=["conceptUri"], keep="first").reset_index(drop=True)

    # Remove rows with no title
    out = out[out["occupation_title"].ne("")].reset_index(drop=True)

    return out[[
        "occupation_id", "conceptUri", "occupation_title", "occupation_key",
        "alt_occupation_title", "occ_description", "isco_group", "occupation_code"
    ]]


# -------------------------
# Standardize skills
# -------------------------
def preprocess_skills(skills_raw):
    sk = skills_raw.copy()
    cols = sk.columns

    concept_col = first_existing(cols, ["conceptUri", "skillUri", "uri"])
    title_col = first_existing(cols, ["preferredLabel", "skill_title", "skillLabel", "skills", "title"])
    alt_col = first_existing(cols, ["altLabels", "alt_skill_title", "alternativeLabels"])
    desc_col = first_existing(cols, ["description", "skill_description"])
    skill_type_col = first_existing(cols, ["skillType", "skill_type"])
    reuse_col = first_existing(cols, ["reuseLevel", "reuse_level"])

    if title_col is None:
        raise ValueError("Skill title column not found.")

    out = pd.DataFrame()
    out["conceptUri"] = sk[concept_col].apply(clean_text) if concept_col else ""
    out["skill_title"] = sk[title_col].apply(clean_label)
    out["skill_key"] = out["skill_title"].apply(normalize_key)
    out["alt_skill_title"] = sk[alt_col].apply(clean_text) if alt_col else ""
    out["skill_description"] = sk[desc_col].apply(clean_text) if desc_col else ""
    out["skill_type"] = sk[skill_type_col].apply(clean_text) if skill_type_col else ""
    out["reuse_level"] = sk[reuse_col].apply(clean_text) if reuse_col else ""

    # If raw file lacks ESCO URI, create stable local URI using skill title.
    missing_uri = out["conceptUri"].eq("")
    out.loc[missing_uri, "conceptUri"] = out.loc[missing_uri, "skill_title"].apply(
        lambda x: "local://skill/" + make_slug_id("skill", x)
    )

    out["skill_id"] = out["conceptUri"].apply(uri_short_id)

    # Fill remaining blanks
    out["alt_skill_title"] = out["alt_skill_title"].fillna("")

    # Drop exact duplicate conceptUri rows
    out = out.drop_duplicates(subset=["conceptUri"], keep="first").reset_index(drop=True)

    # Remove rows with no title
    out = out[out["skill_title"].ne("")].reset_index(drop=True)

    return out[[
        "skill_id", "conceptUri", "skill_title", "skill_key", "alt_skill_title",
        "skill_description", "skill_type", "reuse_level"
    ]]


# -------------------------
# Standardize occupation-skill relations
# -------------------------
def preprocess_relations(rel_raw, occ_clean, skills_clean):
    rel = rel_raw.copy()
    cols = rel.columns

    occ_uri_col = first_existing(cols, ["occupationUri", "conceptUri_occ", "occupation_uri"])
    skill_uri_col = first_existing(cols, ["skillUri", "conceptUri_skill", "skill_uri"])
    occ_label_col = first_existing(cols, ["occupationLabel", "occupation_title", "occupation"])
    skill_label_col = first_existing(cols, ["skillLabel", "skills", "skill_title", "skill"])
    rel_type_col = first_existing(cols, ["relationType", "relation_type"])
    skill_type_col = first_existing(cols, ["skillType", "relation_skill_type"])

    out = pd.DataFrame()
    out["occupationUri"] = rel[occ_uri_col].apply(clean_text) if occ_uri_col else ""
    out["skillUri"] = rel[skill_uri_col].apply(clean_text) if skill_uri_col else ""
    out["occupation_title_raw"] = rel[occ_label_col].apply(clean_label) if occ_label_col else ""
    out["skill_title_raw"] = rel[skill_label_col].apply(clean_label) if skill_label_col else ""
    out["relation_type"] = rel[rel_type_col].apply(clean_text).str.lower() if rel_type_col else "unknown"
    out["relation_skill_type"] = rel[skill_type_col].apply(clean_text) if skill_type_col else ""

    # If URI columns are missing, restore URIs by joining on normalized labels.
    if out["occupationUri"].eq("").all():
        occ_lookup = occ_clean[["occupation_key", "conceptUri"]].drop_duplicates("occupation_key")
        out["occupation_key"] = out["occupation_title_raw"].apply(normalize_key)
        out = out.merge(occ_lookup, on="occupation_key", how="left")
        out["occupationUri"] = out["conceptUri"].fillna("")
        out = out.drop(columns=["conceptUri"])
    
    if out["skillUri"].eq("").all():
        skill_lookup = skills_clean[["skill_key", "conceptUri"]].drop_duplicates("skill_key")
        out["skill_key"] = out["skill_title_raw"].apply(normalize_key)
        out = out.merge(skill_lookup, on="skill_key", how="left")
        out["skillUri"] = out["conceptUri"].fillna("")
        out = out.drop(columns=["conceptUri"])

    # Keep valid relations only
    valid_occ = set(occ_clean["conceptUri"])
    valid_skill = set(skills_clean["conceptUri"])
    before = len(out)
    out = out[out["occupationUri"].isin(valid_occ) & out["skillUri"].isin(valid_skill)].copy()
    invalid_removed = before - len(out)

    out["occupation_id"] = out["occupationUri"].apply(uri_short_id)
    out["skill_id"] = out["skillUri"].apply(uri_short_id)

    # Remove duplicate relation pairs
    dup_before = out.duplicated(subset=["occupationUri", "skillUri"]).sum()
    out = out.drop_duplicates(subset=["occupationUri", "skillUri"], keep="first").reset_index(drop=True)

    # Attach clean labels from master tables
    out = out.merge(
        occ_clean[["conceptUri", "occupation_title"]],
        left_on="occupationUri", right_on="conceptUri", how="left"
    ).drop(columns=["conceptUri"])

    out = out.merge(
        skills_clean[["conceptUri", "skill_title", "skill_type", "reuse_level"]],
        left_on="skillUri", right_on="conceptUri", how="left"
    ).drop(columns=["conceptUri"])

    out["relation_type"] = out["relation_type"].replace("", "unknown")

    return out[[
        "occupation_id", "skill_id", "occupationUri", "skillUri",
        "occupation_title", "skill_title", "relation_type", "skill_type", "reuse_level"
    ]], invalid_removed, int(dup_before)


# -------------------------
# Build combined career roles
# -------------------------
def build_career_roles(occ_clean, rel_clean):
    skill_freq = (
        rel_clean.groupby(["skill_id", "skill_title"])
        .agg(occupation_count=("occupation_id", "nunique"))
        .reset_index()
    )

    total_occupations = occ_clean["occupation_id"].nunique()
    skill_freq["skill_idf_weight"] = np.log((1 + total_occupations) / (1 + skill_freq["occupation_count"])) + 1

    rel2 = rel_clean.merge(skill_freq[["skill_id", "occupation_count", "skill_idf_weight"]], on="skill_id", how="left")

    rel2["relation_priority"] = np.where(rel2["relation_type"].eq("essential"), 0, 1)
    rel2 = rel2.sort_values(
        by=["occupation_id", "relation_priority", "occupation_count", "skill_title"],
        ascending=[True, True, True, True]
    )

    def unique_preserve_order(values):
        return list(dict.fromkeys([v for v in values if isinstance(v, str) and v.strip()]))

    grouped = (
        rel2.groupby("occupation_id")
        .agg(
            all_skills=("skill_title", unique_preserve_order),
            essential_skills=("skill_title", lambda x: unique_preserve_order(rel2.loc[x.index][rel2.loc[x.index, "relation_type"].eq("essential")]["skill_title"])),
            optional_skills=("skill_title", lambda x: unique_preserve_order(rel2.loc[x.index][~rel2.loc[x.index, "relation_type"].eq("essential")]["skill_title"])),
            num_skills=("skill_id", "nunique"),
            avg_skill_weight=("skill_idf_weight", "mean")
        )
        .reset_index()
    )

    grouped["essential_skills_text"] = grouped["essential_skills"].apply(lambda x: "; ".join(x[:MAX_ESSENTIAL_SKILLS_FOR_CONTEXT]))
    grouped["optional_skills_text"] = grouped["optional_skills"].apply(lambda x: "; ".join(x[:MAX_OPTIONAL_SKILLS_FOR_CONTEXT]))
    grouped["skills_list"] = grouped["all_skills"].apply(lambda x: "; ".join(x))

    career = occ_clean.merge(grouped, on="occupation_id", how="left")

    for col in ["skills_list", "essential_skills_text", "optional_skills_text"]:
        career[col] = career[col].fillna("")
    career["num_skills"] = career["num_skills"].fillna(0).astype(int)
    career["avg_skill_weight"] = career["avg_skill_weight"].fillna(0)

    def make_context(row):
        parts = [
            f"Occupation title: {row['occupation_title']}.",
        ]
        if row["alt_occupation_title"]:
            parts.append(f"Alternative titles: {row['alt_occupation_title']}.")
        if row["occ_description"]:
            parts.append(f"Description: {row['occ_description']}.")
        if row["essential_skills_text"]:
            parts.append(f"Essential skills: {row['essential_skills_text']}.")
        if row["optional_skills_text"]:
            parts.append(f"Optional or related skills: {row['optional_skills_text']}.")
        return " ".join(parts)

    career["career_role_context"] = career.apply(make_context, axis=1)

    return career[[
        "occupation_id", "conceptUri", "occupation_title", "alt_occupation_title",
        "occ_description", "isco_group", "occupation_code", "num_skills",
        "skills_list", "essential_skills_text", "optional_skills_text",
        "avg_skill_weight", "career_role_context"
    ]], skill_freq


# -------------------------
# EDA and validation
# -------------------------
def create_eda(occ_raw, skills_raw, rel_raw, occ_clean, skills_clean, rel_clean, career_roles, skill_freq, invalid_removed, dup_rel_removed):
    summary = {
        "raw_occupations_rows": len(occ_raw),
        "clean_occupations_rows": len(occ_clean),
        "raw_skills_rows": len(skills_raw),
        "clean_skills_rows": len(skills_clean),
        "raw_relation_rows": len(rel_raw),
        "clean_relation_rows": len(rel_clean),
        "invalid_relations_removed": invalid_removed,
        "duplicate_relation_pairs_removed": dup_rel_removed,
        "prepared_career_roles": len(career_roles),
        "occupations_with_zero_skills": int((career_roles["num_skills"] == 0).sum()),
        "min_skills_per_occupation": int(career_roles["num_skills"].min()),
        "max_skills_per_occupation": int(career_roles["num_skills"].max()),
        "avg_skills_per_occupation": round(float(career_roles["num_skills"].mean()), 2),
        "median_skills_per_occupation": round(float(career_roles["num_skills"].median()), 2),
        "unused_skills": int((~skills_clean["skill_id"].isin(set(rel_clean["skill_id"]))).sum())
    }

    summary_df = pd.DataFrame(list(summary.items()), columns=["metric", "value"])
    summary_df.to_csv(os.path.join(OUTPUT_DIR, "preprocessing_summary.csv"), index=False)

    missing_report = []
    for name, df in [("occupations_clean", occ_clean), ("skills_clean", skills_clean), ("relations_clean", rel_clean), ("career_roles_prepared", career_roles)]:
        for col, val in df.isna().sum().items():
            missing_report.append({"file": name, "column": col, "missing_count": int(val), "missing_percent": round(float(val / max(len(df), 1) * 100), 4)})
    pd.DataFrame(missing_report).to_csv(os.path.join(OUTPUT_DIR, "missing_value_report.csv"), index=False)

    validation_checks = {
        "no_duplicate_occupation_uri": occ_clean.duplicated("conceptUri").sum() == 0,
        "no_duplicate_skill_uri": skills_clean.duplicated("conceptUri").sum() == 0,
        "no_duplicate_relation_pairs": rel_clean.duplicated(["occupationUri", "skillUri"]).sum() == 0,
        "all_relation_occupation_uris_valid": rel_clean["occupationUri"].isin(set(occ_clean["conceptUri"])).all(),
        "all_relation_skill_uris_valid": rel_clean["skillUri"].isin(set(skills_clean["conceptUri"])).all(),
        "career_context_not_empty": career_roles["career_role_context"].str.len().gt(20).all(),
        "all_occupations_have_skills": (career_roles["num_skills"] > 0).all(),
    }
    validation_df = pd.DataFrame([{"check": k, "passed": bool(v)} for k, v in validation_checks.items()])
    validation_df.to_csv(os.path.join(OUTPUT_DIR, "validation_report.csv"), index=False)

    # Distribution files
    career_roles["num_skills"].describe().to_csv(os.path.join(OUTPUT_DIR, "skills_per_occupation_statistics.csv"))
    rel_clean["relation_type"].value_counts().rename_axis("relation_type").reset_index(name="count").to_csv(
        os.path.join(OUTPUT_DIR, "relation_type_distribution.csv"), index=False
    )

    # Plots
    plt.figure(figsize=(8, 5))
    plt.hist(career_roles["num_skills"], bins=30)
    plt.title("Distribution of Skills per Occupation")
    plt.xlabel("Number of linked skills")
    plt.ylabel("Number of occupations")
    plt.tight_layout()
    plt.savefig(os.path.join(EDA_DIR, "skills_per_occupation_histogram.png"), dpi=150)
    plt.close()

    top20 = skill_freq.sort_values("occupation_count", ascending=False).head(20).sort_values("occupation_count")
    plt.figure(figsize=(8, 7))
    plt.barh(top20["skill_title"], top20["occupation_count"])
    plt.title("Top 20 Most Common ESCO Skills")
    plt.xlabel("Number of occupations")
    plt.ylabel("Skill")
    plt.tight_layout()
    plt.savefig(os.path.join(EDA_DIR, "top20_common_skills.png"), dpi=150)
    plt.close()

    rel_dist = rel_clean["relation_type"].value_counts()
    plt.figure(figsize=(6, 4))
    plt.bar(rel_dist.index.astype(str), rel_dist.values)
    plt.title("Relation Type Distribution")
    plt.xlabel("Relation type")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.savefig(os.path.join(EDA_DIR, "relation_type_distribution.png"), dpi=150)
    plt.close()

    return summary_df, validation_df


# -------------------------
# Save outputs
# -------------------------
def save_outputs(occ_clean, skills_clean, rel_clean, career_roles, skill_freq):
    used_skill_ids = set(rel_clean["skill_id"])
    skills_used = skills_clean[skills_clean["skill_id"].isin(used_skill_ids)].copy()
    unused_skills = skills_clean[~skills_clean["skill_id"].isin(used_skill_ids)].copy()

    occ_clean.to_csv(os.path.join(OUTPUT_DIR, "occupations_clean.csv"), index=False)
    skills_clean.to_csv(os.path.join(OUTPUT_DIR, "skills_clean_all.csv"), index=False)
    skills_used.to_csv(os.path.join(OUTPUT_DIR, "skills_clean_used.csv"), index=False)
    unused_skills.to_csv(os.path.join(OUTPUT_DIR, "unused_skills.csv"), index=False)
    rel_clean.to_csv(os.path.join(OUTPUT_DIR, "occupation_skill_relations_clean.csv"), index=False)
    career_roles.to_csv(os.path.join(OUTPUT_DIR, "career_roles_prepared.csv"), index=False)
    skill_freq.to_csv(os.path.join(OUTPUT_DIR, "skill_frequency.csv"), index=False)

    # JSONL for embedding/indexing
    with open(os.path.join(OUTPUT_DIR, "esco_role_contexts.jsonl"), "w", encoding="utf-8") as f:
        for _, row in career_roles.iterrows():
            f.write(json.dumps({
                "occupation_id": row["occupation_id"],
                "conceptUri": row["conceptUri"],
                "occupation_title": row["occupation_title"],
                "text_for_embedding": row["career_role_context"],
                "num_skills": int(row["num_skills"])
            }, ensure_ascii=False) + "\n")

    data_dictionary = pd.DataFrame([
        ["career_roles_prepared.csv", "occupation_id", "Stable ID extracted from ESCO occupation URI"],
        ["career_roles_prepared.csv", "conceptUri", "Original ESCO occupation URI"],
        ["career_roles_prepared.csv", "occupation_title", "Main ESCO occupation title"],
        ["career_roles_prepared.csv", "alt_occupation_title", "Alternative occupation labels"],
        ["career_roles_prepared.csv", "occ_description", "ESCO occupation description"],
        ["career_roles_prepared.csv", "isco_group", "ISCO group code, if available"],
        ["career_roles_prepared.csv", "occupation_code", "ESCO occupation code, if available"],
        ["career_roles_prepared.csv", "num_skills", "Number of linked ESCO skills"],
        ["career_roles_prepared.csv", "skills_list", "All linked skills"],
        ["career_roles_prepared.csv", "essential_skills_text", "Essential skills selected for embedding context"],
        ["career_roles_prepared.csv", "optional_skills_text", "Optional/related skills selected for embedding context"],
        ["career_roles_prepared.csv", "avg_skill_weight", "Average IDF-style specificity weight of linked skills"],
        ["career_roles_prepared.csv", "career_role_context", "Final combined text used for Gemini embedding and ChromaDB indexing"],
    ], columns=["file", "column", "description"])
    data_dictionary.to_csv(os.path.join(OUTPUT_DIR, "data_dictionary.csv"), index=False)

    return skills_used, unused_skills


# -------------------------
# Main pipeline
# -------------------------
def main():
    occ_raw, skills_raw, rel_raw, source_type = load_esco_files()

    print("\n--- Initial File Review ---")
    review_df(occ_raw, "Occupations raw/current")
    review_df(skills_raw, "Skills raw/current")
    review_df(rel_raw, "Occupation-skill relations raw/current")

    occ_clean = preprocess_occupations(occ_raw)
    skills_clean = preprocess_skills(skills_raw)
    rel_clean, invalid_removed, dup_rel_removed = preprocess_relations(rel_raw, occ_clean, skills_clean)
    career_roles, skill_freq = build_career_roles(occ_clean, rel_clean)

    skills_used, unused_skills = save_outputs(occ_clean, skills_clean, rel_clean, career_roles, skill_freq)
    summary_df, validation_df = create_eda(
        occ_raw, skills_raw, rel_raw, occ_clean, skills_clean, rel_clean,
        career_roles, skill_freq, invalid_removed, dup_rel_removed
    )

    print("\n--- Final Summary ---")
    print(summary_df.to_string(index=False))
    print("\n--- Validation Checks ---")
    print(validation_df.to_string(index=False))

    print("\nFinal prepared dataset sample:")
    display(career_roles.head(5)) if 'display' in globals() else print(career_roles.head(5))

    # Zip all outputs
    zip_name = "processed_esco_dataset"
    shutil.make_archive(zip_name, "zip", OUTPUT_DIR)
    print(f"\nCreated: {zip_name}.zip")
    print("Main file: processed_esco/career_roles_prepared.csv")

    # Colab download helper
    try:
        from google.colab import files
        files.download(zip_name + ".zip")
    except Exception:
        pass


if __name__ == "__main__":
    main()
